# Exploring Knowledge Graphs

## Import packages and set up Neo4

In [1]:
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

# Warning control
import warnings
warnings.filterwarnings("ignore")

/tmp/ipykernel_26537/4289788969.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.graphs import Neo4jGraph


In [2]:
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

* Initialize a knowledge graph instance using LangChain's Neo4j integration

In [3]:
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

## Querying the movie knowledge graph

* Match all nodes in the graph

In [4]:
cypher = """
MATCH (n)
RETURN COUNT(n) AS numberOfNodes
"""

In [5]:
result = kg.query(cypher)
result

[{'numberOfNodes': 114712}]

In [15]:
nodes = kg.query("""
    MATCH (n)
    RETURN n
""")
len(nodes), nodes[:2]

(114712,
 [{'n': {'version_type': 'Original',
    'file_size_gb': 388.64,
    'audio_channels': 'Atmos',
    'title_id': 'TITLE_000558',
    'created_date': '2025-09-11 18:57:18.034831',
    'version_id': 'VER_TITLE_000558_001',
    'is_localized': False,
    'resolution': 'HD',
    'frame_rate': 25.0}},
  {'n': {'version_type': 'Localized',
    'file_size_gb': 184.72,
    'audio_channels': '5.1',
    'title_id': 'TITLE_000558',
    'created_date': '2025-10-24 18:57:18.034837',
    'version_id': 'VER_TITLE_000558_002',
    'is_localized': True,
    'frame_rate': 60.0,
    'resolution': '8K'}}])

In [16]:
indexes = kg.query("""
    SHOW INDEXES
""")
len(indexes), indexes[:2]

(14,
 [{'id': 15,
   'name': 'audio_format_id',
   'state': 'ONLINE',
   'populationPercent': 100.0,
   'type': 'RANGE',
   'entityType': 'NODE',
   'labelsOrTypes': ['AudioFormat'],
   'properties': ['format_id'],
   'indexProvider': 'range-1.0',
   'owningConstraint': 'audio_format_id',
   'lastRead': neo4j.time.DateTime(2026, 5, 23, 21, 19, 6, 196000000, tzinfo=<UTC>),
   'readCount': 72},
  {'id': 7,
   'name': 'client_id',
   'state': 'ONLINE',
   'populationPercent': 100.0,
   'type': 'RANGE',
   'entityType': 'NODE',
   'labelsOrTypes': ['Client'],
   'properties': ['client_id'],
   'indexProvider': 'range-1.0',
   'owningConstraint': 'client_id',
   'lastRead': neo4j.time.DateTime(2026, 5, 23, 21, 19, 15, 165000000, tzinfo=<UTC>),
   'readCount': 221046}])

In [14]:
properties = kg.query("""
    MATCH (n)
    RETURN keys(n)
""")
len(properties), properties[:2]

(114712,
 [{'keys(n)': ['resolution',
    'frame_rate',
    'created_date',
    'version_id',
    'is_localized',
    'file_size_gb',
    'audio_channels',
    'title_id',
    'version_type']},
  {'keys(n)': ['frame_rate',
    'created_date',
    'version_id',
    'resolution',
    'file_size_gb',
    'audio_channels',
    'is_localized',
    'title_id',
    'version_type']}])

In [21]:
properties = kg.query("""
    MATCH (n)
    WITH keys(n) as propertyList
    UNWIND propertyList as property
    RETURN property, count(*) as nodeCount
    ORDER BY nodeCount DESC
""")
len(properties), properties[:2]

(61,
 [{'property': 'version_id', 'nodeCount': 109268},
  {'property': 'client_id', 'nodeCount': 73282}])

In [25]:
schema = kg.query("""
    CALL apoc.meta.schema()
""")
len(schema[0]['value'].keys()), schema[0]['value'].keys()

(21,
 dict_keys(['DeliveryPoint', 'LOCATED_IN', 'DeliveryRequest', 'LocalizationJob', 'Title', 'HAS_VERSION', 'VideoFormat', 'FOR_VERSION', 'FOR_DELIVERY_POINT', 'Rights', 'REQUESTED_BY', 'Language', 'FOR_REGION', 'Version', 'Region', 'TO_POINT', 'Client', 'AudioFormat', 'LOCALIZED_FOR', 'DeliverySpec', 'GRANTED_TO']))

In [32]:
schema = kg.query("""
CALL apoc.meta.schema() YIELD value AS schemaMap
UNWIND keys(schemaMap) AS label
WITH label, schemaMap[label] AS data
WHERE data.type = "node"
UNWIND keys(data.properties) AS property
WITH label, property, data.properties[property] AS propData
RETURN label,
       property,
       propData.type AS type,
       propData.indexed AS isIndexed,
       propData.unique AS uniqueConstraint,
       propData.existence AS existenceConstraint
ORDER BY label, property
""")
len(schema), schema[:2]

(78,
 [{'label': 'AudioFormat',
   'property': 'format_id',
   'type': 'STRING',
   'isIndexed': True,
   'uniqueConstraint': True,
   'existenceConstraint': False},
  {'label': 'AudioFormat',
   'property': 'format_name',
   'type': 'STRING',
   'isIndexed': False,
   'uniqueConstraint': False,
   'existenceConstraint': False}])

In [33]:
set([s['property'] for s in schema])

{'active_since',
 'actual_completion',
 'audio_channels',
 'client_id',
 'client_name',
 'client_type',
 'completion_date',
 'continent',
 'created_at',
 'created_date',
 'credit_limit_usd',
 'deadline',
 'delivery_point_id',
 'delivery_type',
 'duration_minutes',
 'end_date',
 'exclusivity_window_days',
 'file_size_gb',
 'format_id',
 'format_name',
 'frame_rate',
 'genre',
 'hdr_format',
 'is_active',
 'is_localized',
 'is_mandatory',
 'job_id',
 'job_type',
 'language_code',
 'language_family',
 'language_name',
 'max_bitrate_mbps',
 'point_name',
 'priority',
 'quality_score',
 'region_focus',
 'region_id',
 'region_name',
 'release_year',
 'request_date',
 'request_id',
 'required_audio',
 'required_container',
 'required_hdr',
 'required_resolution',
 'resolution',
 'rights_id',
 'rights_type',
 'season_count',
 'spec_id',
 'start_date',
 'status',
 'studio',
 'territorial_restrictions',
 'tier',
 'title_id',
 'title_name',
 'title_type',
 'vendor',
 'version_id',
 'version_type'

In [13]:
cypher = """
MATCH (n:Demo {dataset: 'kg-rag-movie-demo'})
RETURN labels(n), count(*) AS count
ORDER BY count DESC;
"""

In [14]:
kg.query(cypher)

[{'labels(n)': ['Demo', 'Person'], 'count': 30},
 {'labels(n)': ['Demo', 'Theme'], 'count': 19},
 {'labels(n)': ['Movie', 'Demo'], 'count': 10},
 {'labels(n)': ['Demo', 'Genre'], 'count': 5}]

In [15]:
kg.query("""
    MATCH (
        d:Person:Demo {dataset: 'kg-rag-movie-demo', name: 'Christopher Nolan'}
    )-[:DIRECTED]->(
        m:Movie:Demo
    )-[:HAS_GENRE]->(
        :Genre:Demo {dataset: 'kg-rag-movie-demo', name: 'Sci-Fi'}
    )
    MATCH (m)-[:HAS_THEME]->(t:Theme:Demo {dataset: 'kg-rag-movie-demo'})
    WHERE t.name IN ['Memory', 'Time']
    RETURN m.title AS movie,
           m.year AS year,
           collect(DISTINCT t.name) AS matched_themes,
           m.summary AS summary
    ORDER BY year DESC;
""")

[{'movie': 'Tenet',
  'year': 2020,
  'matched_themes': ['Time'],
  'summary': 'An operative manipulates inverted time to prevent global catastrophe in a story driven by temporal mechanics and high-stakes espionage.'},
 {'movie': 'Interstellar',
  'year': 2014,
  'matched_themes': ['Time'],
  'summary': 'A team of explorers travels through a wormhole to find a new home for humanity, where time dilation and sacrifice shape every decision.'},
 {'movie': 'Inception',
  'year': 2010,
  'matched_themes': ['Time', 'Memory'],
  'summary': 'A skilled extractor enters layered dreams to implant an idea while wrestling with guilt, memory, and the unstable nature of reality.'}]